# Polynomial Vector Space Visualization
This notebook demonstrates fundamental linear algebra concepts using polynomial functions:
- Vector space operations (sum, scalar multiplication)
- Derivatives
- Decomposition into basis

All plots use smooth polynomial functions of degree 3.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrow
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgba

import config.config as config

## Define Polynomial Functions and Derivatives

In [ ]:
def p(x): return 1.5 - 1.5 * x - x**2 + 0.7 * x**3
def q(x): return -1 + 2*x - 0.5*x**3
def p_plus_q(x): return p(x) + q(x)
def lambda_p(x): return 2 * p(x)

def delta_p(x): return -1.5 - 2 * x + 2.1 * x**2
def delta_q(x): return 2 - 1.5 * x**2
def delta_sum(x): return delta_p(x) + delta_q(x)
def delta_lambda_p(x): return 2 * delta_p(x)


## Helper Functions for Gradient Arrows

In [ ]:
def gradient_arrow(ax, x0, y0, y1, color_start, color_end, n_segments=20):
    ys = np.linspace(y0, y1, n_segments + 1)
    segments = [((x0, ys[i]), (x0, ys[i + 1])) for i in range(n_segments)]
    colors = [blend_colors(color_start, color_end, i / n_segments) for i in range(n_segments)]
    lc = LineCollection(segments, colors=colors, linewidths=1.3, zorder=3)
    ax.add_collection(lc)
    dx, dy = 0, y1 - y0
    if dy != 0:
        head = FancyArrow(x0, y1 - 0.01 * dy, dx, 0.01 * dy, width=0.015,
                          head_width=0.05, head_length=0.15,
                          length_includes_head=True,
                          color=color_end, zorder=4)
        ax.add_patch(head)

def blend_colors(c1, c2, t):
    return tuple((1 - t) * np.array(c1) + t * np.array(c2))


## Plot: Vector Operations and Derivatives

In [ ]:
x = np.linspace(-1, 1, 400)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# --- Operations Plot
ax = ax1
ax.plot(x, p(x), label=r"$p(x) = 1.5 - 1.5x - x^2 + 0.7x^3$", color='royalblue', lw=2.5)
ax.plot(x, q(x), label=r"$q(x) = -1 + 2x - \frac{1}{2}x^3$", color='forestgreen', lw=2.5)
ax.plot(x, p_plus_q(x), label=r"$p \oplus q$", color='orange', lw=2.5)
ax.plot(x, lambda_p(x), color='darkorchid', lw=2.5, label=r"$2 \odot p(x)$")

spacing = np.linspace(-0.8, 0.8, 6)
for xi in spacing:
    idx = np.abs(x - xi).argmin()
    gradient_arrow(ax, x[idx], p(x)[idx], p_plus_q(x)[idx], to_rgba('royalblue'), to_rgba('orange'))
    gradient_arrow(ax, x[idx], q(x)[idx], p_plus_q(x)[idx], to_rgba('forestgreen'), to_rgba('orange'))

spacing = np.linspace(-0.9, 0.9, 8)
for xi in spacing:
    idx = np.abs(x - xi).argmin()
    gradient_arrow(ax, x[idx], p(x)[idx], lambda_p(x)[idx], to_rgba('royalblue'), to_rgba('darkorchid'))

ax.set_xlim(-1, 1)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Polynomial Operations")
ax.legend()

# --- Derivative Plot
ax = ax2
ax.plot(x, delta_p(x), label=r"$\delta(p)(x)$", color='royalblue', lw=2.5)
ax.plot(x, delta_q(x), label=r"$\delta(q)(x)$", color='forestgreen', lw=2.5)
ax.plot(x, delta_sum(x), label=r"$\delta(p \oplus q)$", color='orange', lw=2.5)
ax.plot(x, delta_lambda_p(x), label=r"$\delta(2 \odot p)$", color='darkorchid', lw=2.5)

spacing = np.linspace(-0.8, 0.8, 6)
for xi in spacing:
    idx = np.abs(x - xi).argmin()
    gradient_arrow(ax, x[idx], delta_p(x)[idx], delta_sum(x)[idx], to_rgba('royalblue'), to_rgba('orange'))
    gradient_arrow(ax, x[idx], delta_q(x)[idx], delta_sum(x)[idx], to_rgba('forestgreen'), to_rgba('orange'))

spacing = np.linspace(-0.9, 0.9, 8)
for xi in spacing:
    idx = np.abs(x - xi).argmin()
    gradient_arrow(ax, x[idx], delta_p(x)[idx], delta_lambda_p(x)[idx], to_rgba('royalblue'), to_rgba('darkorchid'))

ax.set_xlim(-1, 1)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Derivative Linear Properties")
ax.legend()

fig.tight_layout()
fig.savefig(os.path.join(config.save_dir, "vector_space_sum_and_prod.pdf"))
plt.show()


## Plot: Decomposition of $p(x)$ and $q(x)$ into Basis

In [ ]:
p_coeffs = [1.5, -1.5, -1.0, 0.7]
q_coeffs = [-1, 2, 0, -0.5]

def e0(x): return np.ones_like(x)
def e1(x): return x
def e2(x): return x**2
def e3(x): return x**3
basis_funcs = [e0, e1, e2, e3]
basis_labels = [r"$e_0(x) = 1$", r"$e_1(x) = x$", r"$e_2(x) = x^2$", r"$e_3(x) = x^3$"]
basis_linestyles = ['-', '--', '-.', ':']

def p(x): return sum(c * f(x) for c, f in zip(p_coeffs, basis_funcs))
def q(x): return sum(c * f(x) for c, f in zip(q_coeffs, basis_funcs))

x = np.linspace(-1, 1, 400)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

# --- Panel for p(x)
p_lines = []
for i, (c, f) in enumerate(zip(p_coeffs, basis_funcs)):
    if c != 0:
        line, = ax1.plot(x, c * f(x), linestyle=basis_linestyles[i], lw=2.2, color='royalblue',
                         label=fr"${c} \cdot {basis_labels[i][1:4]}$")
        p_lines.append(line)
p_main = ax1.plot(x, p(x), color='royalblue', lw=2.5, label=r"$p(x)$")

basis_lines = []
for i, f in enumerate(basis_funcs):
    line, = ax1.plot(x, f(x), linestyle=basis_linestyles[i], color='gray', lw=1.2, alpha=0.7,
                     label=basis_labels[i])
    basis_lines.append(line)

ax1.set_title(r"Decomposition of $p(x)$")
ax1.set_xlabel(r"$x$")
ax1.set_ylabel(r"$y$")
ax1.set_xlim(-1, 1)
first_legend = ax1.legend(handles=p_lines + p_main, loc='lower right', fontsize=9, title="Function and components")
second_legend = ax1.legend(handles=basis_lines, loc='lower left', fontsize=9, title="Basis functions")
ax1.add_artist(first_legend)

# --- Panel for q(x)
q_lines = []
for i, (c, f) in enumerate(zip(q_coeffs, basis_funcs)):
    if c != 0:
        line, = ax2.plot(x, c * f(x), linestyle=basis_linestyles[i], lw=2.2, color='forestgreen',
                         label=fr"${c} \cdot {basis_labels[i][1:4]}$")
        q_lines.append(line)
q_main = ax2.plot(x, q(x), color='forestgreen', lw=2.5, label=r"$q(x)$")

basis_lines = []
for i, f in enumerate(basis_funcs):
    line, = ax2.plot(x, f(x), linestyle=basis_linestyles[i], color='gray', lw=1.2, alpha=0.7,
                     label=basis_labels[i])
    basis_lines.append(line)

ax2.set_title(r"Decomposition of $q(x)$")
ax2.set_xlabel(r"$x$")
ax2.set_xlim(-1, 1)
first_legend_q = ax2.legend(handles=q_lines + q_main, loc='lower right', fontsize=9, title="Function and components")
ax2.add_artist(first_legend_q)

fig.tight_layout()
fig.savefig(os.path.join(config.save_dir, "vector_space_decomposition.pdf"))
plt.show()
